# CineScope — Hit Prediction (Spark MLlib)

**Local Spark only.** Trains a GBT classifier for provisional hit label:
`average_rating >= 7.0 AND num_votes >= 1000`.

Features are **pre-release only** (no rating/votes/Oscar columns).  
Confusion matrix is shown inline and saved under `outputs/charts/generated/`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
load_dotenv(REPO / ".env")

from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

from cinescope.ml.features import (
    available_pre_release_columns,
    build_feature_pipeline,
    leakage_notes,
    prepare_modeling_frame,
)
from cinescope.paths import get_paths
from cinescope.spark_session import build_spark_session

paths = get_paths(create_dirs=True, validate_mount=True)
spark = build_spark_session(app_name="cinescope-train-hit", paths=paths)
spark.sparkContext.setLogLevel("WARN")

metrics_dir = REPO / "outputs" / "metrics"
charts_dir = REPO / "outputs" / "charts" / "generated"
metrics_dir.mkdir(parents=True, exist_ok=True)
charts_dir.mkdir(parents=True, exist_ok=True)
model_dir = paths.data_root / "models" / "hit_gbt"

def save_and_show(fig, path: Path):
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", path)

raw = spark.read.parquet(str(paths.movies_awards_enriched_dir))
df = prepare_modeling_frame(raw)
feature_cols = available_pre_release_columns(df)
print("feature_cols", feature_cols)
print("rows", df.count(), "hit_rate", df.agg(F.avg("is_hit")).first()[0])

In [ ]:
for c in feature_cols:
    df = df.withColumn(c, F.col(c).cast("double"))

train, test = df.randomSplit([0.8, 0.2], seed=42)
pipe = build_feature_pipeline(feature_cols, label_col="is_hit")
gbt = GBTClassifier(
    labelCol="is_hit",
    featuresCol="features",
    maxIter=40,
    maxDepth=5,
    seed=42,
)
model = Pipeline(stages=pipe.getStages() + [gbt]).fit(train)
pred = model.transform(test)

auc = BinaryClassificationEvaluator(
    labelCol="is_hit", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(pred)
acc = MulticlassClassificationEvaluator(
    labelCol="is_hit", predictionCol="prediction", metricName="accuracy"
).evaluate(pred)
f1 = MulticlassClassificationEvaluator(
    labelCol="is_hit", predictionCol="prediction", metricName="f1"
).evaluate(pred)

tp = pred.filter((F.col("is_hit") == 1) & (F.col("prediction") == 1)).count()
fp = pred.filter((F.col("is_hit") == 0) & (F.col("prediction") == 1)).count()
fn = pred.filter((F.col("is_hit") == 1) & (F.col("prediction") == 0)).count()
tn = pred.filter((F.col("is_hit") == 0) & (F.col("prediction") == 0)).count()
precision = tp / (tp + fp) if (tp + fp) else None
recall = tp / (tp + fn) if (tp + fn) else None
print("AUC", auc, "accuracy", acc, "f1", f1)
print("precision", precision, "recall", recall)
print("tn, fp, fn, tp", tn, fp, fn, tp)

In [ ]:
cm = np.array([[tn, fp], [fn, tp]], dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1], ["pred 0", "pred 1"])
axes[0].set_yticks([0, 1], ["actual 0", "actual 1"])
axes[0].set_title("Hit model confusion matrix")
for (i, j), v in np.ndenumerate(cm):
    axes[0].text(j, i, int(v), ha="center", va="center", color="black")
fig.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].bar(
    ["AUC", "precision", "recall", "accuracy"],
    [auc, precision or 0, recall or 0, acc],
    color=["C0", "C2", "C3", "C1"],
)
axes[1].set_ylim(0, 1)
axes[1].set_title("Hit model metrics")
axes[1].grid(True, axis="y", alpha=0.3)
fig.tight_layout()
save_and_show(fig, charts_dir / "hit_model_confusion_metrics.png")

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("### Hit model — results summary"))
display(pd.DataFrame(
    [
        ("Algorithm", "GBTClassifier"),
        ("Label", "Hit: rating ≥ 7.0 and votes ≥ 1000"),
        ("Environment", "local Spark notebook"),
        ("Train rows", train.count()),
        ("Test rows", test.count()),
        ("AUC", round(auc, 4)),
        ("Precision", None if precision is None else round(precision, 4)),
        ("Recall", None if recall is None else round(recall, 4)),
        ("F1", round(f1, 4)),
        ("Accuracy", round(acc, 4)),
    ],
    columns=["Metric", "Value"],
))

display(Markdown("### Confusion matrix (test set)"))
display(pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actual 0", "Actual 1"],
    columns=["Pred 0", "Pred 1"],
))

display(Markdown("### Notes"))
display(Markdown(
    "- Hits are a small class — prefer **AUC / precision / recall** over accuracy.\n"
    "- Features are pre-release only (no rating/votes/Oscar columns) — see `leakage_notes()`."
))
display(pd.DataFrame({"Leakage note": leakage_notes()}))


In [ ]:
model.write().overwrite().save(str(model_dir))
metrics = {
    "job": "train_hit_model",
    "model_path": str(model_dir),
    "algorithm": "GBTClassifier",
    "feature_cols": feature_cols,
    "leakage": leakage_notes(),
    "train_rows": train.count(),
    "test_rows": test.count(),
    "metrics": {
        "auc": auc,
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    },
    "charts": [str(charts_dir / "hit_model_confusion_metrics.png")],
    "environment": "local Spark notebook",
}
out = metrics_dir / "hit_model_metrics.json"
out.write_text(json.dumps(metrics, indent=2))
print("Saved model + metrics file for the report:", out)
spark.stop()